In [1]:
import pandas as pd
import numpy as np

In [2]:
paths = ({
    'cleaned_dataset': '../data/cleaned dataset.parquet',
    'invoice_level_table': '../data/D3_invoice-level table.csv'
})

In [3]:
df = pd.read_parquet(paths['cleaned_dataset'])
df.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


## Part A: feature engineering techniques (vectorization, binning, conditional column, map)

#### 1. create `Revenue` column: vectorized column arithmetic

In [4]:
# revenue
df_revenue = df.copy()

df_revenue['Revenue'] = df_revenue['UnitPrice'] * df_revenue['Quantity']
df_revenue.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00


#### 2. binary flag columm (eg: big order)

In [5]:
df_bigorder = df_revenue.copy()
df_bigorder['BigOrder'] = np.where(df_bigorder['Revenue'] > 5000, 1, 0) 
df_bigorder[df_bigorder['BigOrder']==1]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,BigOrder
32727,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,6539.40,1
37120,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60,1
109613,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,6539.40,1
118341,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,8142.75,1
155405,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,38970.00,1
248685,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450.0,United Kingdom,7144.72,1
397411,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60,1


#### 3. month column derived from datetime

In [6]:
df_month = df_bigorder.copy()

import datetime

df_month['InvoiceDate'] = pd.to_datetime(df_month['InvoiceDate'])
df_month['Month'] = df_month['InvoiceDate'].dt.month
df_month.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,BigOrder,Month
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,0,12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0,12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,0,12


#### 4. binned revenue column using `cut` or `qcut`
Change continuous variable to categorical variable

In [7]:
df_bin = df_month.copy()

# cut: cut by numerical cut-offs
df_bin['cut_bins'] = pd.cut(df_bin['Revenue'], 
                       bins=4, 
                       labels=['a', 'b', 'c', 'd'],
                       retbins=False, # return bins boundaries as an array, 
                                       # True will return tuple of (Series, array)
                       duplicates= 'drop' # If bin edges not unique, drop non-uniques
                                       # default will raise ValueError
                      )
df_bin.sort_values(['cut_bins']).tail(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,BigOrder,Month,cut_bins
397883,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,14.85,0,12,a
37120,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60,1,1,b
397411,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60,1,12,d


In [8]:
# qcut: split by quantile
df_bin['cutq_bins'] = pd.qcut(df_bin['Revenue'], 
                              q=4,
                              labels=['I', 'II', 'III', 'IV']
                             )
df_bin.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,BigOrder,Month,cut_bins,cutq_bins
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,0,12,a,III
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0,12,a,IV
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,0,12,a,IV


#### 5. map: elementwise functions if unable to vectorize 

In [9]:
df_map = df_bin.copy()

def length(x):
    return len(str(x))

df_map['LengthOfText'] = df_map['Description'].map(length)
df_map.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,BigOrder,Month,cut_bins,cutq_bins,LengthOfText
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,0,12,a,III,34
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0,12,a,IV,19
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,0,12,a,IV,30


## Part B: agg stats table

#### 1. Invoice-level aggregation

In [10]:
df_invoice = df_map.groupby('InvoiceNo').agg(
    invoice_first_date=('InvoiceDate', 'min'),
    customer_id=('CustomerID', 'first'),
    country=('Country', 'first'),
    invoice_revenue=('Revenue', 'sum'),
    n_lines=('InvoiceNo', 'count')
).reset_index()
df_invoice

,InvoiceNo,invoice_first_date,customer_id,country,invoice_revenue,n_lines
0,536365,2010-12-01 08:26:00,17850.0,United Kingdom,139.12,7
1,536366,2010-12-01 08:28:00,17850.0,United Kingdom,22.20,2
2,536367,2010-12-01 08:34:00,13047.0,United Kingdom,278.73,12
3,536368,2010-12-01 08:34:00,13047.0,United Kingdom,70.05,4
4,536369,2010-12-01 08:35:00,13047.0,United Kingdom,17.85,1
...,...,...,...,...,...,...
18527,581583,2011-12-09 12:23:00,13777.0,United Kingdom,124.60,2
18528,581584,2011-12-09 12:25:00,13777.0,United Kingdom,140.64,2
18529,581585,2011-12-09 12:31:00,15804.0,United Kingdom,329.05,21
18530,581586,2011-12-09 12:49:00,13113.0,United Kingdom,339.20,4


In [11]:
df_invoice.to_csv(paths['invoice_level_table'], index=False)